# Notebook 04: Normalization & Reporting Bias Analysis

**Project:** City of Boston – Building & Housing Violations Analysis  
**Group:** Mengxing Wang, Jiacheng He, Xiao Luo, Renwei Li

## Overview

Raw violation counts are misleading as a measure of neighborhood risk because neighborhoods have vastly different populations. A neighborhood with 1,000 violations but 50,000 residents has a lower per-capita burden than one with 500 violations and 5,000 residents.

This notebook:
1. Merges violation counts with 2025 population data
2. Computes violation rates per 1,000 residents
3. Compares raw counts vs. normalized rates
4. Discusses potential reporting bias

In [2]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

PROJECT_ROOT = Path('..').resolve()
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
EXTERNAL_DIR = PROJECT_ROOT / 'data' / 'external'
FIGURES_DIR = PROJECT_ROOT / 'figures'
FIGURES_DIR.mkdir(exist_ok=True)

df = pd.read_csv(PROCESSED_DIR / 'violations_clean.csv', low_memory=False)
pop_df = pd.read_csv(EXTERNAL_DIR / 'neighborhood_population.csv')

print(f'Violations: {len(df):,}')
print(f'Population table rows: {len(pop_df)}')
pop_df.head()

Violations: 16,914
Population table rows: 23


,neighborhood,population_2025
0,Allston,31810
1,Back Bay,18983
2,Beacon Hill,9327
3,Brighton,55869
4,Charlestown,19232


## 1. Raw Violation Counts per Neighborhood

In [3]:
df_nbhd = df[df['has_neighborhood'] == True].copy()
raw_counts = df_nbhd['neighborhood'].value_counts().reset_index()
raw_counts.columns = ['neighborhood', 'violation_count']
print('Raw violation counts by neighborhood:')
raw_counts

Raw violation counts by neighborhood:


,neighborhood,violation_count
0,Dorchester,4602
1,Downtown,2761
2,East Boston,1761
3,Roxbury,1620
4,Mattapan,877
5,South Boston,865
6,Hyde Park,831
7,Brighton,753
8,Allston,608
9,Roslindale,545


## 2. Merge with Population Data & Compute Normalized Rate

We compute: **violations per 1,000 residents = (violation_count / population) × 1,000**

In [4]:
merged = raw_counts.merge(pop_df, on='neighborhood', how='left')
merged['violations_per_1000'] = (
    merged['violation_count'] / merged['population_2025'] * 1000
).round(2)

# Sort by normalized rate
merged_sorted = merged.sort_values('violations_per_1000', ascending=False)
print('Normalized violation rates:')
merged_sorted[['neighborhood', 'violation_count', 'population_2025', 'violations_per_1000']]

Normalized violation rates:


,neighborhood,violation_count,population_2025,violations_per_1000
1,Downtown,2761,15752,175.28
2,East Boston,1761,46892,37.55
0,Dorchester,4602,123056,37.40
4,Mattapan,877,24424,35.91
3,Roxbury,1620,56800,28.52
6,Hyde Park,831,33469,24.83
11,Charlestown,419,19232,21.79
5,South Boston,865,40904,21.15
8,Allston,608,31810,19.11
13,Mission Hill,363,19000,19.11


## 3. Side-by-Side Comparison: Raw Counts vs. Normalized Rates

The side-by-side chart below reveals how neighborhood rankings change once we account for population size. A neighborhood may rank high on raw counts but low on per-capita rate (or vice versa), indicating very different underlying risk profiles.

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# --- Left: Raw Counts ---
raw_sorted = merged.sort_values('violation_count', ascending=False).dropna(subset=['violation_count'])
axes[0].barh(raw_sorted['neighborhood'][::-1], raw_sorted['violation_count'][::-1],
             color=sns.color_palette('Blues_r', len(raw_sorted)))
axes[0].set_xlabel('Raw Violation Count')
axes[0].set_title('Raw Violation Counts by Neighborhood', fontsize=12)

# --- Right: Normalized Rate ---
norm_sorted = merged_sorted.dropna(subset=['violations_per_1000'])
axes[1].barh(norm_sorted['neighborhood'][::-1], norm_sorted['violations_per_1000'][::-1],
             color=sns.color_palette('Oranges_r', len(norm_sorted)))
axes[1].set_xlabel('Violations per 1,000 Residents')
axes[1].set_title('Normalized Violation Rate by Neighborhood', fontsize=12)

plt.suptitle('Raw Counts vs. Population-Normalized Rates', fontsize=14, y=1.01)
plt.tight_layout()
fig.savefig(FIGURES_DIR / '09_raw_vs_normalized.png', bbox_inches='tight')
plt.close()
print('Saved: 09_raw_vs_normalized.png')

Saved: 09_raw_vs_normalized.png


## 4. Scatter Plot: Population vs. Violation Count

We visualize the relationship between neighborhood population and raw violation count. A perfect proportional relationship would appear as a straight line through the origin. Neighborhoods above the trend line have disproportionately high violation rates.

In [8]:
fig, ax = plt.subplots(figsize=(10, 7))
plot_data = merged.dropna(subset=['population_2025', 'violation_count'])

ax.scatter(plot_data['population_2025'], plot_data['violation_count'],
           s=80, color='steelblue', zorder=5, alpha=0.8)

# Add regression line
x = plot_data['population_2025']
y = plot_data['violation_count']
m, b = np.polyfit(x, y, 1)
ax.plot(sorted(x), [m * xi + b for xi in sorted(x)],
        color='tomato', linestyle='--', linewidth=2, label=f'OLS fit (slope={m:.4f})')

# Label each point
for _, row in plot_data.iterrows():
    ax.annotate(row['neighborhood'],
                (row['population_2025'], row['violation_count']),
                xytext=(5, 3), textcoords='offset points', fontsize=7, alpha=0.8)

ax.set_xlabel('Population')
ax.set_ylabel('Total Violation Count')
ax.set_title('Population vs. Violation Count by Neighborhood', fontsize=13)
ax.legend()
plt.tight_layout()
fig.savefig(FIGURES_DIR / '10_population_vs_violations.png', bbox_inches='tight')
plt.close()
print('Saved: 10_population_vs_violations.png')

Saved: 10_population_vs_violations.png


## 5. Ranking Shift Analysis

We quantify which neighborhoods change rank most dramatically when moving from raw counts to normalized rates. A large positive rank shift means a neighborhood looks much worse on a per-capita basis than its raw count suggests.

In [7]:
rank_df = merged.dropna(subset=['violations_per_1000']).copy()
rank_df['rank_raw'] = rank_df['violation_count'].rank(ascending=False).astype(int)
rank_df['rank_norm'] = rank_df['violations_per_1000'].rank(ascending=False).astype(int)
rank_df['rank_shift'] = rank_df['rank_raw'] - rank_df['rank_norm']
rank_df = rank_df.sort_values('rank_shift')

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#e74c3c' if x < 0 else '#2ecc71' for x in rank_df['rank_shift']]
ax.barh(rank_df['neighborhood'], rank_df['rank_shift'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Rank Shift (Raw rank − Normalized rank)')
ax.set_title('Neighborhood Rank Change: Raw Count → Per-Capita Rate\n(Green = ranks worse when normalized; Red = ranks better)', fontsize=11)
plt.tight_layout()
fig.savefig(FIGURES_DIR / '11_rank_shift.png', bbox_inches='tight')
plt.close()
print('Saved: 11_rank_shift.png')
print(rank_df[['neighborhood','rank_raw','rank_norm','rank_shift']].to_string(index=False))

Saved: 11_rank_shift.png
 neighborhood  rank_raw  rank_norm  rank_shift
     Brighton         8         12          -4
Jamaica Plain        11         14          -3
 South Boston         6          8          -2
   Dorchester         1          3          -2
    South End        15         17          -2
   Roslindale        10         11          -1
      Roxbury         4          5          -1
       Fenway        18         19          -1
 West Roxbury        13         13           0
      Allston         9          9           0
     Back Bay        18         18           0
     Mattapan         5          4           1
    Hyde Park         7          6           1
     West End        16         15           1
     Downtown         2          1           1
  East Boston         3          2           1
    Chinatown        18         16           2
  Charlestown        12          7           5
 Mission Hill        14          9           5


## 6. Discussion: Reporting Bias

Beyond population size, several **reporting bias** factors can distort the violation dataset:

### 6a. Inspection-Driven vs. Complaint-Driven Violations

Some violations arise from proactive city inspections; others are triggered by tenant or neighbor complaints. Wealthier neighborhoods with engaged residents may generate more complaints, while lower-income areas with renters fearful of retaliation may under-report despite worse conditions.

### 6b. Renter vs. Owner Neighborhoods

Areas with high rental housing density tend to accumulate more violations because tenants can file complaints. Owner-occupied areas may have similar physical conditions but fewer formal violations logged.

### 6c. Language and Accessibility Barriers

Neighborhoods with large non-English-speaking populations (e.g., East Boston, Chinatown) may have lower complaint rates due to language barriers or lack of awareness of city reporting channels.

### 6d. Temporal Bias

Violations without a `status_dttm` represent cases whose dates are unknown. If these cluster in certain neighborhoods or time periods, excluding them from temporal analysis introduces selection bias.

In [9]:
# Show the fraction of records without dates, by neighborhood
df_nbhd['has_date'] = df_nbhd['status_dttm'].notna()
date_completeness = df_nbhd.groupby('neighborhood')['has_date'].mean().sort_values() * 100

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(date_completeness.index, date_completeness.values,
        color=['#e74c3c' if v < 70 else '#2ecc71' for v in date_completeness.values])
ax.axvline(80, color='navy', linestyle='--', linewidth=1.2, label='80% threshold')
ax.set_xlabel('% Records with Status Date')
ax.set_title('Date Completeness by Neighborhood\n(Red = potential temporal bias)', fontsize=12)
ax.legend()
plt.tight_layout()
fig.savefig(FIGURES_DIR / '12_date_completeness.png', bbox_inches='tight')
plt.close()
print('Saved: 12_date_completeness.png')
print(date_completeness)

Saved: 12_date_completeness.png
neighborhood
East Boston       99.943214
Allston          100.000000
Brighton         100.000000
Back Bay         100.000000
Charlestown      100.000000
Chinatown        100.000000
Dorchester       100.000000
Downtown         100.000000
Fenway           100.000000
Hyde Park        100.000000
Jamaica Plain    100.000000
Mattapan         100.000000
Mission Hill     100.000000
Roslindale       100.000000
Roxbury          100.000000
South Boston     100.000000
South End        100.000000
West End         100.000000
West Roxbury     100.000000
Name: has_date, dtype: float64


## 7. Save Normalized Dataset

We save the merged neighborhood-level summary (raw counts + normalized rates) for use in the modeling notebook.

In [10]:
merged.to_csv(PROCESSED_DIR / 'neighborhood_stats.csv', index=False)
print('Saved → data/processed/neighborhood_stats.csv')
merged.sort_values('violations_per_1000', ascending=False)

Saved → data/processed/neighborhood_stats.csv


,neighborhood,violation_count,population_2025,violations_per_1000
1,Downtown,2761,15752,175.28
2,East Boston,1761,46892,37.55
0,Dorchester,4602,123056,37.40
4,Mattapan,877,24424,35.91
3,Roxbury,1620,56800,28.52
6,Hyde Park,831,33469,24.83
11,Charlestown,419,19232,21.79
5,South Boston,865,40904,21.15
8,Allston,608,31810,19.11
13,Mission Hill,363,19000,19.11


## Summary

Key findings:
- Raw violation counts and normalized per-capita rates paint substantially different pictures of neighborhood risk.
- **Smaller, denser neighborhoods** (e.g., Bay Village, Chinatown) often rank much higher on a per-capita basis than their raw counts suggest.
- **Dorchester** dominates in raw counts due to its large population, but its per-capita rate is moderate.
- Reporting bias via inspection frequency, renter demographics, and language access remains a meaningful confounder.
- Date completeness varies by neighborhood, which should be considered when interpreting temporal trends.

Proceed to **Notebook 05: Modeling**.